# tools

In [20]:
from dotenv import load_dotenv
load_dotenv()

True

## 1. 自定义工具
- 工具名
- 工具的作用
- 工具的参数

### 1.1 基于tool描述工具

In [1]:
from langchain_core.tools import tool

@tool("square_root", description="计算一个数的平方根")
def square_root(x: float) -> float:
    return x ** 0.5


### 1.2 使用函数名加文档注释描述工具

In [2]:
from langchain_core.tools import tool

@tool
def square_root(x: float) -> float:
    """计算一个数的平方根"""
    return x ** 0.5

In [11]:
square_root.invoke({"x": 14})

3.7416573867739413

In [ ]:
import os
import requests
from langchain_core.tools import tool

# 获取天气信息的工具函数
@tool
def get_weather(location: str, date: str = "明天") -> str:
    """
    获取特定日期和地点的天气信息，并返回一个格式化的字符串。
    参数:
    - location: 天气发生的地点
    - date: 天气发生的日期（默认为“明天”）
    """
    try:
        # 高德地图天气 API（免费申请）
        AMAP_KEY = os.getenv("GAODE_TIANQI_API_KEY")  # 等下我教你免费拿
        url = "https://restapi.amap.com/v3/weather/weatherInfo"
        
        # 1. 先获取城市编码
        city_url = "https://restapi.amap.com/v3/config/district"
        city_params = {
            "keywords": location,
            "key": AMAP_KEY,
            "subdistrict": 0,
            "output": "json"
        }
        city_resp = requests.get(city_url, params=city_params).json()
        if not city_resp.get("districts"):
            return f"未找到城市：{city}"
        city_code = city_resp["districts"][0]["adcode"]

        # 2. 获取天气
        weather_params = {
            "city": city_code,
            "key": AMAP_KEY,
            "extensions": "all",  # 获取预报
            "output": "json"
        }
        resp = requests.get(url, params=weather_params).json()
        forecasts = resp["forecasts"][0]["casts"]
        # 3. 返回对应日期天气
        if date == "今天":
            data = forecasts[0]
        elif date == "明天":
            data = forecasts[1]
        elif date == "后天":
            data = forecasts[2]
        else:
            data = forecasts[1]

        return (
            f"{date} {city} 天气：{data['dayweather']}，"
            f"温度 {data['nighttemp']}~{data['daytemp']}℃，"
            f"风向：{data['daywind']}，风力：{data['daypower']}级"
        )
    except Exception as e:
        return f"天气查询失败：{str(e)}"

### 1.3 pyantic model定义参数
- 当参数比较多或者比较复杂时

In [6]:
from pydantic import BaseModel,Field
from langchain_core.tools import tool
from typing import Literal

class WeatherResponse(BaseModel):
    units: Literal["Celsius", "Fahrenheit"] = Field(default="Celsius", description="温度单位")
    location: str = Field(..., description="天气发生的地点")
    include_forecast: bool = Field(default=False, description="是否包含未来几天的天气预报")

@tool(args_schema=WeatherResponse)
def get_weather(location: str, units: str = "Celsius", include_forecast: bool = False) -> dict:
    """
    获取特定地点的天气信息，并返回一个结构化的响应。
    参数:
    - location: 天气发生的地点
    - units: 温度单位（默认为“Celsius”）
    - include_forecast: 是否包含未来几天的天气预报（默认为False）
    """
    try:
        # 这里我们模拟一个天气API的响应
        weather_data = {
            "location": location,
            "temperature": 25 if units == "Celsius" else 77,
            "condition": "晴朗",
            "forecast": [
                {"date": "2024-06-01", "condition": "多云", "high": 28, "low": 20},
                {"date": "2024-06-02", "condition": "小雨", "high": 22, "low": 18},
            ] if include_forecast else []
        }
        return weather_data
    except Exception as e:
        return {"error": f"天气查询失败：{str(e)}"}



In [15]:
from langchain_classic.agents import (
    AgentExecutor,
    create_openai_tools_agent,
)
from langchain_classic import hub
from langchain_openai import ChatOpenAI

prompt = hub.pull("hwchase17/openai-tools-agent")

llm = ChatOpenAI(
    model="qwen-plus",
    api_key=os.getenv("DASHSCOPE_API_KEY"), # type: ignore
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1", 
)

tools = [get_weather, square_root]

agent = create_openai_tools_agent(llm, tools, prompt)

agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

response = agent_executor.invoke({
    "input": "请帮我查询一下明天杭州的天气"
})

print("\n✅ 最终回答：", response["output"])



> Entering new AgentExecutor chain...

Invoking: `get_weather` with `{'location': '杭州', 'include_forecast': True}`


{'location': '杭州', 'temperature': 25, 'condition': '晴朗', 'forecast': [{'date': '2024-06-01', 'condition': '多云', 'high': 28, 'low': 20}, {'date': '2024-06-02', 'condition': '小雨', 'high': 22, 'low': 18}]}根据查询结果，明天（2024年6月1日）杭州的天气预报如下：

- 天气状况：多云  
- 最高温度：28°C  
- 最低温度：20°C  

此外，后天（2024年6月2日）将有小雨，气温在18°C至22°C之间。当前杭州天气晴朗，气温为25°C。如需进一步帮助，请随时告诉我！

> Finished chain.

✅ 最终回答： 根据查询结果，明天（2024年6月1日）杭州的天气预报如下：

- 天气状况：多云  
- 最高温度：28°C  
- 最低温度：20°C  

此外，后天（2024年6月2日）将有小雨，气温在18°C至22°C之间。当前杭州天气晴朗，气温为25°C。如需进一步帮助，请随时告诉我！


In [16]:
response = agent_executor.invoke({
    "input": "345的平方根是多少？"
})

print("\n✅ 最终回答：", response["output"])



> Entering new AgentExecutor chain...

Invoking: `square_root` with `{'x': 345}`


18.57417562100671345的平方根约为18.574。

> Finished chain.

✅ 最终回答： 345的平方根约为18.574。


In [14]:
get_weather.invoke({
    "location": "杭州",
    "units": "Celsius",
    "include_forecast": True
})

{'location': '杭州',
 'temperature': 25,
 'condition': '晴朗',
 'forecast': [{'date': '2024-06-01', 'condition': '多云', 'high': 28, 'low': 20},
  {'date': '2024-06-02', 'condition': '小雨', 'high': 22, 'low': 18}]}

## 2. 预定义工具

In [18]:
from tavily import TavilyClient

tavily_client = TavilyClient(api_key="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")
response = tavily_client.search("介绍一下李嘉诚?")

print(response)

{'query': '介绍一下李嘉诚?', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://www.mrbaike.cn/nd.jsp?id=187', 'title': '李嘉诚 - 名人百科', 'content': '|  | 李嘉诚 二维码 20249  李嘉诚，男，汉族，1928年7月29日出生于广东潮州潮安 ，祖籍福建莆田，现任长江和记实业有限公司及长江实业地产有限公司主席。  1958年，李嘉诚开始投资地产市场。1979年，“长江”购入老牌英资商行——“和记黄埔”，李嘉诚因而成为首位收购英资商行的华人。1981年获选“香港风云人物”和太平绅士，1989年获英国女王颁发的CBE勋衔，1992年被聘为港事顾问，1993年度香港风云人物，1995年至1997年任特区筹备委员会委员，自从1999年被福布斯评为全球华人首富以来，连续15年蝉联华人首富宝座。2011年工夫茶传奇故事入选国家孔子学院汉语外教文章。2015年8月19日，胡润发布2015年全球华人富豪榜，李嘉诚以2000亿元财富位列全球华人富豪第二位。2018年2月28日，胡润研究院发布《2018胡润全球富豪榜》，李嘉诚财富1900亿元位列大中华区第六，全球第32位。  2018年3月16日，满90岁的李嘉诚正式宣布退休。5月10日，出席集团股东大会，之后正式退休。2018年6月29日，汕头大学2018年毕业典礼之后辞去汕头大学校董事会名誉主席职务，交棒给二儿子李泽楷。  1928年7月29日，李嘉诚生于广东潮州潮安县北门街面线巷的一个教师之家。5岁那年，李嘉诚在父亲李云经的引导下，祭拜孔子儒学，进入观海寺小学念书。  1939年6月，刚刚读初中的李嘉诚在与家人辗转到香港，一家人寄居在舅父庄静庵的家里。  1941年12月8日，太平洋战争爆发，圣诞节前夕，香港英军向日军投降。港币不断贬值，物价飞涨，李家生活愈加困难，而父亲李云经又在这时因劳累过度染上肺病。  **职业生涯**  1943年冬天，父亲李云经病逝，为了养活母亲和三个弟妹，14岁的李嘉诚被迫辍学走上社会谋生。李嘉诚先在舅父庄静庵的中南钟表公司当泡茶扫地的小学徒。李嘉诚到这里之后，学到的第一个功夫就是察言观色，见机行事

In [ ]:
from langchain_tavily import TavilySearch
from langchain.agents import create_agent

# Initialize Tavily Search Tool
tavily_search_tool = TavilySearch(
    max_results=5,
    topic="general",
)

agent = create_agent(
    model="deepseek-chat",
    tools=[tavily_search_tool],  # 空工具列表
    system_prompt="你是一个知识渊博的助手，能够使用Tavily Search工具来获取最新的信息。请根据用户的查询使用Tavily Search工具进行搜索，并提供准确和相关的回答。"
)

user_input = "介绍一下周杰伦?"

for step in agent.stream(
    {"messages": user_input},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()


================================ Human Message =================================

介绍一下周杰伦?
================================== Ai Message ==================================

我来搜索一下关于周杰伦的最新信息。
Tool Calls:
  tavily_search (call_00_AWMcRdfRaHcVj2FLzf0Ncwjo)
 Call ID: call_00_AWMcRdfRaHcVj2FLzf0Ncwjo
  Args:
    query: 周杰伦 简介 生平 音乐成就
    search_depth: advanced
================================= Tool Message =================================
Name: tavily_search

{"query": "周杰伦 简介 生平 音乐成就", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://baike.baidu.com/item/%E5%91%A8%E6%9D%B0%E4%BC%A6/129156", "title": "周杰伦（华语流行乐男歌手、音乐人、演员、导演）_百度百科", "content": "本    名\n:   周杰伦\n\n外文名\n:   Jay Chou\n\n昵    称\n:   周董、小公举\n\n性    别\n:   男\n\n民    族\n:   汉族\n\n国    籍\n:   中国\n\n出生地\n:   台湾省新北市\n\n出生日期\n:   1979年1月18日\n\n毕业院校\n:   淡江中学\n\n星    座\n:   摩羯座\n\n血    型\n:   O型\n\n身    高\n:   175 cm\n\n体    重\n:   60 kg\n\n成名作品\n:   音乐专辑《Jay》\n\n经纪公司\n:   杰威尔音乐有限公司\n\n粉丝名称\n:   杰

### 2.1 自定义tavily工具

In [23]:
from langchain_tavily import TavilySearch
from langchain_core.tools import tool

# Initialize Tavily Search Tool
tavily_search_tool = TavilySearch(
    max_results=5,
    topic="general",
)

@tool
def tavily_search(query: str) -> str:
    """
    使用Tavily Search工具进行搜索，并返回搜索结果的摘要。
    参数:
    - query: 用户的查询字符串
    """
    try:
        results = tavily_search_tool.invoke({"query": query})
        # 这里我们简单地返回搜索结果的标题和链接
        summary = "\n".join([f"{res['title']}: {res['url']}" for res in results])
        return summary
    except Exception as e:
        return f"Tavily Search查询失败：{str(e)}"

### 2.2 定义结构化输出主体

In [24]:
from pydantic import BaseModel, Field
#agent回答引用的网页信息
class Reference(BaseModel):
    title: str = Field(..., description="网页标题")
    url: str = Field(..., description="网页链接")
    snippet: str = Field(..., description="网页摘要")

#agent回答内容
class AgentAnswer(BaseModel):
    content: str = Field(..., description="agent的回答内容")
    references: list[Reference] = Field(default_factory=list, description="agent回答引用的网页信息列表")

In [ ]:
from langchain.agents import create_agent
agent = create_agent(
    model="deepseek-chat",
    tools=[tavily_search_tool],  # 空工具列表
    system_prompt="你是一个知识渊博的助手，能够使用Tavily Search工具来获取最新的信息。请根据用户的查询使用Tavily Search工具进行搜索，并提供准确和相关的回答。",
    response_format=AgentAnswer
)

user_input = "介绍一下周杰伦?"

for step in agent.stream(
    {"messages": user_input},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

介绍一下周杰伦?
================================== Ai Message ==================================
Tool Calls:
  tavily_search (call_00_JSBELH0e7U1Wh7HGD2ShMXWU)
 Call ID: call_00_JSBELH0e7U1Wh7HGD2ShMXWU
  Args:
    query: 周杰伦 个人资料 介绍
    search_depth: advanced
================================= Tool Message =================================
Name: tavily_search

{"query": "周杰伦 个人资料 介绍", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://www.chinadaily.com.cn/hqylss/2006-03/03/content_526325.htm", "title": "周杰伦个人资料", "content": "|  | |  |  |  |  |  |  |  |  |  ---  ---  ---  --- | | |  |  |  |  |  |  ---  ---  | |  |  | 周杰伦个人资料  [ 2006-03-03 16:59 ] | | 周杰伦个人资料中文姓名: 周杰伦  英文姓名: Jay Chou  生　日: 1979-01-18  出生地: 台湾  身　高: 173cm  体　重: 60kg  血　型: O  星　座: 摩羯座  学 历: 淡江中学音乐科  专 长: 写歌、作词、打球。  专精乐器:钢琴、大提琴、吉他、爵士鼓。  音乐类型:R&B  运 动 :篮球  最大优点:词曲创作力强、e世代的新音乐人。  最大缺点:害羞、心软。  最大心愿:走遍世界每一个角落  最爱物品:鸭

In [30]:
response = agent.invoke(
    {"messages": user_input},
)

print("\n✅ 最终回答：", response['structured_response'])


✅ 最终回答： content='## 周杰伦简介\n\n**周杰伦**（英语：Jay Chou，1979年1月18日—），出生于中国台湾省新北市（原台北县），祖籍福建省泉州市永春县，是华语乐坛最具影响力的歌手、词曲创作人、音乐制作人、演员和导演之一。\n\n### 主要成就与影响力\n\n- **亚洲天王**：他的音乐风行于大中华地区及全球华人社群，深刻影响了华语流行音乐的发展，形成独特的"周杰伦现象"。\n- **时代杂志封面**：2003年成为美国《时代周刊》亚洲版封面人物，被誉为"亚洲流行音乐的新天王"。\n- **CNN评选**：2009年被CNN评选为"亚洲最有影响力人物之一"。\n\n### 音乐生涯\n\n- **出道**：2000年发行首张同名专辑《Jay》正式出道，立即引起华语乐坛轰动。\n- **风格特点**：融合R&B、嘻哈、中国风等多种音乐元素，歌词常由搭档方文山创作，作品具有强烈的个人风格和辨识度。\n- **经典专辑**：截至2023年已发行20多张个人专辑，代表作品包括《七里香》《叶惠美》《十一月的萧邦》《魔杰座》《范特西》等。\n\n### 其他领域\n\n- 参演过多部电影，如《头文字D》《不能说的秘密》《满城尽带黄金甲》等。\n- 自导自演电影《不能说的秘密》，展现导演才华。\n- 曾担任广播节目和电视节目主持人。\n\n周杰伦的音乐陪伴了无数人的青春，是华语乐坛一个时代的标志性人物，至今仍拥有巨大的影响力和粉丝基础。' references=[]
